# WoodFisher

Метод WoodFisher (NeurIPS 2020, [«WoodFisher: Efficient Second-Order Approximation for Neural Network Compression»](https://arxiv.org/abs/2004.14340)) — это практически применимый способ использовать информацию второго порядка для прунинга нейронных сетей с миллионами параметров. Идейно WoodFisher продолжает линию Optimal Brain Surgeon (OBS), но решает её главную проблему: невозможность работать с полным гессианом для больших моделей.

В этом разделе мы построим метод с нуля: начнём с задачи OBS, разберёмся, какие приближения и почему делает WoodFisher, выведем все ключевые формулы и обсудим практические детали реализации.

### Review of OBS approach

Пусть $\theta \in \mathbb{R}^d$ — обученные веса сети, $\mathcal{L}(\theta)$ — функция потерь. Раскладываем изменение лосса при возмущении весов $\delta\theta$ в ряд Тейлора:

$
\delta \mathcal{L} = g^\top \delta\theta + \tfrac{1}{2} \delta\theta^\top H \, \delta\theta + O(\|\delta\theta\|^3),
$

где $g = \nabla \mathcal{L}(\theta)$ — градиент, $H = \nabla^2 \mathcal{L}(\theta)$ — гессиан.

Предположим, что сеть обучена до (локального) минимума: $g \approx 0$. Тогда:

$
\delta \mathcal{L} \approx \tfrac{1}{2} \, \delta\theta^\top H \, \delta\theta.
$

Хотим удалить $q$-й вес, то есть наложить ограничение $\theta_q + \delta\theta_q = 0$, или эквивалентно $e_q^\top \delta\theta = -\theta_q$, где $e_q$ — стандартный базисный вектор. Остальные веса разрешено корректировать, чтобы компенсировать удаление и минимизировать рост лосса:

$
\min_{\delta\theta} \; \tfrac{1}{2} \, \delta\theta^\top H \, \delta\theta \quad \text{при условии} \quad e_q^\top \delta\theta + \theta_q = 0.
$

Составим лагранжиан:

$
\Lambda = \tfrac{1}{2} \delta\theta^\top H \, \delta\theta + \lambda \, (e_q^\top \delta\theta + \theta_q).
$

Условия первого порядка:

$
\frac{\partial \Lambda}{\partial \delta\theta} = H \, \delta\theta + \lambda \, e_q = 0 \;\Longrightarrow\; \delta\theta = -\lambda \, H^{-1} e_q.
$

Подставляем в ограничение:

$
e_q^\top (-\lambda H^{-1} e_q) = -\theta_q \;\Longrightarrow\; \lambda = \frac{\theta_q}{[H^{-1}]_{qq}}.
$

Отсюда — две ключевые формулы OBS:

$
\boxed{\; \delta\theta = -\frac{\theta_q}{[H^{-1}]_{qq}} \, H^{-1} e_q, \qquad s_q = \tfrac{1}{2} \cdot \frac{\theta_q^2}{[H^{-1}]_{qq}}. \;}
$

Здесь $s_q$ — saliency веса $q$ (рост лосса при его оптимальном удалении), а $\delta\theta$ — оптимальное обновление всех весов.

#### Почему OBS не масштабируется

Для сети с $d$ параметрами:

- $H$ имеет размер $d \times d$ — для ResNet-50 ($d \approx 25 \cdot 10^6$) это $\sim 2.5 \cdot 10^{15}$ элементов, около 10 ПБ в FP32.
- $H^{-1}$ требует ещё больше: явное обращение стоит $O(d^3)$.
- Даже хранение одной строки $H^{-1} e_q$ — это $d$ чисел, и нужно $d$ таких строк, чтобы оценить все saliency.

Optimal Brain Damage обошла это, отбросив все недиагональные элементы $H$. OBS — наоборот, нужна вся обратная матрица. WoodFisher ищет промежуточный путь.

### От гессиана к информации Фишера

Прямая работа с $H$ для нейросетей проблематична по нескольким причинам:

- $H$ может быть индефинитной (не положительно определённой), особенно для невыпуклых задач, и тогда формула $\delta\mathcal{L} = \tfrac{1}{2}\delta\theta^\top H \delta\theta \geq 0$ даже не гарантирует роста лосса.
- Вычисление произведения $Hv$ требует двойного обратного распространения и шумно.
- Для классификации лосс — это negative log-likelihood, и здесь есть естественная PSD-замена.

### Эмпирическая матрица Фишера

Пусть модель задаёт условное распределение $p_\theta(y \mid x)$, а $\mathcal{L} = -\log p_\theta(y \mid x)$. Тогда определяется матрица Фишера:

$$
F = \mathbb{E}_{x \sim \mathcal{D}, \, y \sim p_\theta(\cdot \mid x)} \left[ \nabla_\theta \log p_\theta(y \mid x) \, \nabla_\theta \log p_\theta(y \mid x)^\top \right].
$$

На практике используется *эмпирическая* версия — с реальными метками $y_i$ и средним по обучающим (или калибровочным) данным:

$$
\hat F = \frac{1}{N} \sum_{i=1}^N g_i \, g_i^\top, \qquad g_i = \nabla_\theta \log p_\theta(y_i \mid x_i).
$$

Ключевые свойства $\hat F$:

- Всегда положительно полуопределена (сумма внешних произведений).
- При условии корректно специфицированной модели и оптимума: $F \approx H$ (классический результат из теории максимального правдоподобия).
- Вычисляется через стандартный backprop — без вторых производных.

WoodFisher заменяет $H$ на $\hat F$ во всех формулах OBS:

$$
s_q \approx \tfrac{1}{2} \cdot \frac{\theta_q^2}{[\hat F^{-1}]_{qq}}, \qquad \delta\theta \approx -\frac{\theta_q}{[\hat F^{-1}]_{qq}} \, \hat F^{-1} e_q.
$$

Замена обоснована эмпирически: на практике $\hat F$ оказывается достаточно хорошим прокси для $H$ в окрестности обученной точки.

### Ядро метода: формула Шермана–Моррисона

Главное наблюдение WoodFisher: $\hat F$ — это сумма ранг-1 матриц. Для таких сумм существует рекуррентная формула обращения, не требующая хранения самой матрицы.

### Формула Шермана–Моррисона

Пусть $A$ — обратимая матрица, $u, v \in \mathbb{R}^d$. Тогда:

$$
(A + uv^\top)^{-1} = A^{-1} - \frac{A^{-1} u v^\top A^{-1}}{1 + v^\top A^{-1} u}.
$$

#### Вывод

Обозначим $B = A + uv^\top$ и $C = A^{-1} - \dfrac{A^{-1} u v^\top A^{-1}}{1 + v^\top A^{-1} u}$. Покажем, что $BC = I$. Распишем:

$$
BC = (A + uv^\top)\left( A^{-1} - \frac{A^{-1} u v^\top A^{-1}}{1 + v^\top A^{-1} u} \right).
$$

Раскрываем:

$$
BC = I + uv^\top A^{-1} - \frac{A A^{-1} u v^\top A^{-1} + u v^\top A^{-1} u v^\top A^{-1}}{1 + v^\top A^{-1} u}.
$$

В числителе второго слагаемого вынесем $u$ и $v^\top A^{-1}$:

$$
A A^{-1} u v^\top A^{-1} + u v^\top A^{-1} u v^\top A^{-1} = u v^\top A^{-1} + u (v^\top A^{-1} u) v^\top A^{-1} = u (1 + v^\top A^{-1} u) v^\top A^{-1}.
$$

Скаляр $1 + v^\top A^{-1} u$ сокращается со знаменателем:

$$
BC = I + uv^\top A^{-1} - u v^\top A^{-1} = I. \quad \blacksquare
$$

#### Применение к матрице Фишера

Запишем эмпирическую матрицу Фишера рекуррентно. Введём регуляризацию $\lambda I$ для гарантированной обратимости (это эквивалентно prior $\mathcal{N}(0, \lambda^{-1})$ на веса):

$$
F_0 = \lambda I, \qquad F_n = F_{n-1} + \tfrac{1}{N} g_n g_n^\top, \qquad \hat F = F_N.
$$

Применяем формулу Шермана–Моррисона на каждом шаге к $F_n^{-1}$:

$$
\boxed{\; F_n^{-1} = F_{n-1}^{-1} - \frac{F_{n-1}^{-1} g_n g_n^\top F_{n-1}^{-1}}{N + g_n^\top F_{n-1}^{-1} g_n}. \;}
$$

(Множитель $N$ в знаменателе появляется из-за $\tfrac{1}{N}$ перед $g_n g_n^\top$.)

Это и есть «вудберовское» обновление, давшее название методу: Sherman–Morrison — частный случай формулы Вудбери для ранг-1 апдейтов.

#### Алгоритмическая сложность

- Хранение $F_n^{-1}$: $O(d^2)$ — всё ещё неприемлемо для больших $d$.
- Одно обновление: $O(d^2)$ — нужно посчитать $F_{n-1}^{-1} g_n$ и внешнее произведение.

Без дополнительных идей мы пока выиграли только избавление от явного обращения $\hat F$ — но не от квадратичной памяти. Следующий шаг — блочно-диагональное приближение.

### Блочно-диагональное приближение

WoodFisher не пытается работать с полной матрицей $\hat F^{-1}$ размера $d \times d$. Вместо этого она аппроксимируется блочно-диагональной матрицей:

$$
\hat F \approx \mathrm{blockdiag}(\hat F^{(1)}, \hat F^{(2)}, \ldots, \hat F^{(B)}),
$$

где каждый блок $\hat F^{(b)} \in \mathbb{R}^{b \times b}$ соответствует группе из $b$ подряд идущих параметров (обычно весов одного слоя или его части). Размер блока $b$ — гиперпараметр; типичные значения от нескольких сотен до нескольких тысяч.

#### Зачем именно блоки

- Память: вместо $O(d^2)$ нужно $O(d \cdot b)$ — линейно по $d$.
- Время одного обновления: $O(b^2)$ вместо $O(d^2)$.
- Параллелизм: блоки обрабатываются независимо.
- Точность: внутри слоя корреляции градиентов сильнее, чем между удалёнными слоями, так что отбрасывание дальних недиагональных элементов — разумная эвристика.

### Структура хранения

Для каждого блока $b$ поддерживается матрица $\hat F_{(b)}^{-1} \in \mathbb{R}^{b \times b}$. Обновления выполняются независимо: для $n$-го примера берём соответствующую блочную часть градиента $g_n^{(b)} \in \mathbb{R}^b$ и применяем Sherman–Morrison к этому блоку:

$$
[F_n^{(b)}]^{-1} = [F_{n-1}^{(b)}]^{-1} - \frac{[F_{n-1}^{(b)}]^{-1} g_n^{(b)} (g_n^{(b)})^\top [F_{n-1}^{(b)}]^{-1}}{N + (g_n^{(b)})^\top [F_{n-1}^{(b)}]^{-1} g_n^{(b)}}.
$$

### Что мы теряем

Блочно-диагональное приближение игнорирует кросс-блочные взаимодействия: компенсация при удалении веса в слое $l$ не распространяется на слой $l+1$. Singh & Alistarh показали, что для типичных архитектур это приемлемая цена за вычислительную осуществимость

### Алгоритм WoodFisher целиком

Соберём всё вместе.

#### Вход

- Обученная модель с весами $\theta$.
- Калибровочный набор данных размера $N$ (обычно от 1000 до 10000 примеров, не нужен полный train set).
- Размер блока $b$.
- Регуляризация $\lambda > 0$ (типично $10^{-5}$–$10^{-3}$).
- Целевая степень разреженности или число удаляемых весов $k$.

#### Шаг 1: оценка обратной Фишер-матрицы

Для каждого блока $b$:

1. Инициализировать $\hat F_{(b)}^{-1} \leftarrow \lambda^{-1} I_b$.
2. Для $n = 1, \ldots, N$:
   - Выполнить forward+backward на примере $(x_n, y_n)$, получить градиент $g_n$.
   - Извлечь блочный фрагмент $g_n^{(b)}$.
   - Обновить $\hat F_{(b)}^{-1}$ по формуле Шермана–Моррисона.

#### Шаг 2: вычисление saliency

Для каждого веса $q$ в блоке $b$:

$$
s_q = \tfrac{1}{2} \cdot \frac{\theta_q^2}{[\hat F_{(b)}^{-1}]_{qq}}.
$$

#### Шаг 3: выбор весов для удаления

Сортируем все $s_q$ глобально или послойно, выбираем $k$ весов с наименьшими saliency. Получаем индексное множество $\mathcal{Q}$.

#### Шаг 4: компенсирующее обновление

Здесь у WoodFisher есть тонкость: формула OBS выводилась для удаления *одного* веса, а удалить нужно сразу множество. Singh & Alistarh используют приближённое решение — суммирование индивидуальных обновлений:

$$
\delta\theta \approx -\sum_{q \in \mathcal{Q}} \frac{\theta_q}{[\hat F^{-1}]_{qq}} \, \hat F^{-1} e_q.
$$

Это не точное решение задачи многомерного OBS (для точного нужно решать систему $|\mathcal{Q}| \times |\mathcal{Q}|$, что снова дорого), но на практике работает хорошо. Точное решение для блока возможно и иногда применяется в более поздних работах.

Эквивалентная форма: $\delta\theta$ — это столбцы $\hat F^{-1}$, соответствующие удаляемым индексам, взвешенные коэффициентами $-\theta_q / [\hat F^{-1}]_{qq}$.

#### Шаг 5: применение

Обнуляем веса с индексами из $\mathcal{Q}$, прибавляем $\delta\theta$ к остальным. Опционально — короткое дообучение для восстановления качества.

### Итеративная версия

В оригинальной работе показано, что лучшие результаты даёт итеративный (gradual) прунинг: удалять веса небольшими порциями, между порциями обновляя $\hat F^{-1}$ на текущих весах. Это компенсирует то, что Фишер-матрица меняется при изменении $\theta$.

### Корректное многомерное обобщение

Для полноты — как выглядит точная задача OBS при удалении сразу $|\mathcal{Q}| = m$ весов.

Ограничения: $E_\mathcal{Q}^\top (\theta + \delta\theta) = 0$, где $E_\mathcal{Q} \in \mathbb{R}^{d \times m}$ — матрица из столбцов $e_q$ для $q \in \mathcal{Q}$. Эквивалентно: $E_\mathcal{Q}^\top \delta\theta = -\theta_\mathcal{Q}$.

Лагранжиан: $\Lambda = \tfrac{1}{2} \delta\theta^\top \hat F \delta\theta + \mu^\top (E_\mathcal{Q}^\top \delta\theta + \theta_\mathcal{Q})$.

Условия первого порядка: $\hat F \delta\theta + E_\mathcal{Q} \mu = 0 \Rightarrow \delta\theta = -\hat F^{-1} E_\mathcal{Q} \mu$.

Подставляем в ограничение:

$$
-E_\mathcal{Q}^\top \hat F^{-1} E_\mathcal{Q} \mu = -\theta_\mathcal{Q} \;\Longrightarrow\; \mu = \big( E_\mathcal{Q}^\top \hat F^{-1} E_\mathcal{Q} \big)^{-1} \theta_\mathcal{Q}.
$$

Откуда точные формулы:

$$
\delta\theta = -\hat F^{-1} E_\mathcal{Q} \big( [\hat F^{-1}]_{\mathcal{Q}\mathcal{Q}} \big)^{-1} \theta_\mathcal{Q},
$$

$$
s_\mathcal{Q} = \tfrac{1}{2} \, \theta_\mathcal{Q}^\top \big( [\hat F^{-1}]_{\mathcal{Q}\mathcal{Q}} \big)^{-1} \theta_\mathcal{Q}.
$$

Здесь $[\hat F^{-1}]_{\mathcal{Q}\mathcal{Q}} \in \mathbb{R}^{m \times m}$ — подматрица $\hat F^{-1}$ по индексам $\mathcal{Q}$. Видно, что приближение «сумма независимых апдейтов», использованное в WoodFisher, точно тогда и только тогда, когда $[\hat F^{-1}]_{\mathcal{Q}\mathcal{Q}}$ диагональна — то есть когда взаимодействиями между удаляемыми весами можно пренебречь.

Внутри блока размера $b$ обращение $m \times m$ подматрицы доступно, и продвинутые варианты метода это используют. Это же наблюдение лежит в основе SparseGPT, где послойная задача решается *точно* колонка за колонкой.

### Связь с другими методами

| Метод | Что аппроксимируется | Структура аппроксимации |
|---|---|---|
| Magnitude pruning | — | saliency $= \|\theta_q\|$ |
| Optimal Brain Damage | $H$ | диагональ |
| Optimal Brain Surgeon | $H^{-1}$ | полная |
| K-FAC | $F$ | Кронекеровская факторизация послойно |
| WoodFisher | $F^{-1}$ | блочно-диагональная, итеративная через Шермана–Моррисона |
| M-FAC | $F^{-1}$ | через формулу Вудбери в матричной форме, без блоков |
| SparseGPT | послойный $H$ | точная развязка через update колонок |

WoodFisher и M-FAC (Frantar et al., 2021) — близкие методы; M-FAC выводит более эффективную форму через матричную (не последовательную) формулу Вудбери и оптимизирует под GPU.

### Практические замечания

- Калибровочный набор данных. Несколько тысяч примеров обычно достаточно; больше — диминишн. Важно, чтобы выборка покрывала распределение задачи.
- Выбор размера блока. Большие блоки точнее, но дороже по памяти и времени. На практике подбирается под слой: для свёрток блок может совпадать с фильтром, для FC — с отдельной строкой или малой группой строк.
- Регуляризация $\lambda$. Слишком малая — численная нестабильность (особенно когда градиенты почти линейно зависимы); слишком большая — приближение скатывается к magnitude pruning, потому что $\hat F^{-1} \to \lambda^{-1} I$ и saliency пропорциональна $\theta_q^2$.
- Стабильность Шермана–Моррисона. При большом $N$ накапливается численная погрешность. На практике помогают: использование FP64 при обновлениях, периодический пересчёт с нуля, или формулировка через факторизацию (хранение $L$ вместо $\hat F^{-1}$).
- Сравнение с magnitude. На умеренных степенях разреженности (до 50–60%) выигрыш WoodFisher над хорошо настроенным iterative magnitude pruning часто невелик. По-настоящему метод оправдывает себя на высоких разреженностях и в задачах one-shot прунинга без длинного дообучения.

### References

- Singh, Alistarh. *WoodFisher: Efficient Second-Order Approximation for Neural Network Compression* (NeurIPS 2020) — оригинальная статья.
- Hassibi, Stork. *Second Order Derivatives for Network Pruning: Optimal Brain Surgeon* (NeurIPS 1992) — исходная формулировка OBS.
- Martens, Grosse. *Optimizing Neural Networks with Kronecker-factored Approximate Curvature* (ICML 2015) — K-FAC, альтернативный способ структурировать $F$.
- Frantar, Kurtic, Alistarh. *M-FAC: Efficient Matrix-Free Approximations of Second-Order Information* (NeurIPS 2021) — дальнейшее развитие идеи.
- Frantar, Alistarh. *Optimal Brain Compression* (NeurIPS 2022) и *SparseGPT* (ICML 2023) — масштабирование на LLM через послойную постановку.